# T02 — Squidpy: Spatial Transcriptomics Analysis

**Tool:** `squidpy`  
**Dataset:** Mouse brain Visium H&E (10x Genomics public dataset)  
**Paper:** [Palla et al. 2022, Nature Methods](https://doi.org/10.1038/s41592-021-01358-2)

---

## What is spatial transcriptomics?

Standard scRNA-seq loses **where** each cell is in the tissue. Spatial methods preserve coordinates:

| Platform | Resolution | Throughput |
|----------|-----------|------------|
| 10x Visium | ~55 µm spots (1–10 cells) | ~4,000 spots / slide |
| Visium HD | ~2 µm (sub-cellular) | ~11M bins |
| Slide-seq v2 | ~10 µm | ~50k beads |
| MERFISH / Xenium | single-cell, in situ | ~100–500 genes |
| seqFISH+ | single-cell, in situ | ~10k genes |

## What does Squidpy add?

Scanpy handles the gene expression side. Squidpy adds:
- **Spatial graph construction** (neighbors based on physical proximity)
- **Neighborhood enrichment** — which cell types are co-localized?
- **Spatial autocorrelation** — which genes vary spatially? (Moran's I)
- **Co-occurrence** — enrichment at a range of spatial distances
- **Ligand-receptor interactions** in spatial context
- **Image features** — extract morphological features from H&E images

The SpatialData framework (`spatialdata`) is the newer unified container for spatial data (multi-scale images, shapes, labels). Squidpy works with both AnnData+coordinates and SpatialData objects.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'squidpy {sq.__version__}')

## 1. Load Spatial Data

The spatial coordinate information is stored in `adata.obsm['spatial']` — a numpy array of shape `(n_spots, 2)` with (x, y) pixel coordinates. The H&E image goes in `adata.uns['spatial']`.

In [ ]:
# Built-in Visium mouse brain dataset (~2,695 spots, 18,078 genes)
# Downloads ~100 MB on first run, cached to ~/.cache/squidpy
adata = sq.datasets.visium_hne_adata()
print(adata)
print('\nobs columns:', adata.obs.columns.tolist())
print('uns keys:', list(adata.uns.keys()))
print('obsm keys:', list(adata.obsm.keys()))

In [ ]:
# The spatial coordinates
print('Spatial coords shape:', adata.obsm['spatial'].shape)
print('First 3 spots:\n', adata.obsm['spatial'][:3])

In [ ]:
# Visualize spots on the H&E tissue image
# cluster = pre-annotated tissue region labels included in the dataset
sq.pl.spatial_scatter(
    adata,
    color='cluster',
    title='Mouse brain Visium — annotated clusters',
    figsize=(6, 6),
)

In [ ]:
# Overlay specific genes on the tissue
# Good for quick sanity-check of known brain markers
sq.pl.spatial_scatter(
    adata,
    color=['Nrgn', 'Penk', 'Mbp'],  # neurons, enkephalin, myelin
    size=1.5,
    figsize=(12, 4),
    ncols=3,
)

## 2. Standard Gene Expression Processing

Same scanpy pipeline as T00 — squidpy just adds the spatial layer on top.

In [ ]:
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.4)

sc.pl.umap(adata, color=['leiden', 'cluster'], legend_loc='on data')

## 3. Spatial Neighborhood Graph

Instead of a kNN graph in gene expression space, we build one in **physical space**. `sq.gr.spatial_neighbors` builds a graph where each spot is connected to its physically adjacent spots.

In [ ]:
# Build spatial kNN graph
# coord_type='visium' uses the Visium grid geometry (hexagonal packing)
# n_neighs=6 = 6 nearest tissue neighbors
sq.gr.spatial_neighbors(adata, coord_type='visium', n_neighs=6)

# The graph is stored in adata.obsp['spatial_connectivities']
# and adata.obsp['spatial_distances']
print('Spatial graph:', adata.obsp['spatial_connectivities'])

## 4. Neighborhood Enrichment

Ask: *are cell type A and cell type B found adjacent to each other more (or less) often than by chance?*

Permutation test: shuffle cell type labels → compute observed co-occurrence / shuffled distribution → z-score.

In [ ]:
# Uses adata.obsp['spatial_connectivities'] + adata.obs[cluster_key]
# n_perms=500 for real use; 50 here for speed
sq.gr.nhood_enrichment(adata, cluster_key='cluster', n_perms=50, seed=42)
print('Result stored in:', list(adata.uns['cluster_nhood_enrichment'].keys()))

In [ ]:
sq.pl.nhood_enrichment(
    adata,
    cluster_key='cluster',
    title='Neighborhood enrichment (z-score)',
    figsize=(8, 6),
)

## 5. Co-occurrence Analysis

Neighborhood enrichment is binary (adjacent / not). Co-occurrence measures enrichment across **a range of spatial distances** — does cell type A appear near B more than expected, and at what radius?

In [ ]:
# Subsample for speed (co-occurrence is O(n²))
adata_sub = adata[::5].copy()  # every 5th spot

sq.gr.co_occurrence(adata_sub, cluster_key='cluster')
sq.pl.co_occurrence(
    adata_sub,
    cluster_key='cluster',
    clusters='Hippocampus',   # focus on one cluster
    figsize=(10, 4),
)

## 6. Spatial Autocorrelation (Moran's I)

Moran's I asks: *is this gene's expression spatially patterned, or randomly distributed?*
- Moran's I ≈ 1 → strongly spatial (nearby spots have similar expression)
- Moran's I ≈ 0 → spatially random
- Moran's I < 0 → negative spatial autocorrelation (checkerboard)

Useful for discovering spatially variable genes without using cluster labels.

In [ ]:
# Compute Moran's I for all highly variable genes
# Uses the spatial_connectivities graph
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    genes=adata.var_names[adata.var['highly_variable']],
    n_perms=50,
    n_jobs=1,
)
# Results in adata.uns['moranI']
moranI = adata.uns['moranI'].sort_values('I', ascending=False)
print('Top 10 spatially variable genes:')
print(moranI.head(10)[['I', 'pval_norm']])

In [ ]:
# Visualize the top spatially variable genes
top_svgs = moranI.head(6).index.tolist()
sq.pl.spatial_scatter(
    adata,
    color=top_svgs,
    ncols=3,
    figsize=(12, 8),
    title=[f'{g}\n(I={moranI.loc[g, "I"]:.2f})' for g in top_svgs],
)

## 7. Ligand-Receptor Interactions

Squidpy can test whether ligand-expressing spots are physically adjacent to receptor-expressing spots, using a database of known LR pairs (CellChat, NATMI, etc.).

In [ ]:
# Ligand-receptor permutation test on spatial graph
# Uses the Ligand-Receptor database from CellChatDB by default
# n_perms=100 is minimum useful; use 1000 for publication
res = sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key='cluster',
    copy=True,
    use_raw=False,
    transmitter_params={'categories': 'ligand'},
    receiver_params={'categories': 'receptor'},
)

# res is a dict with 'means', 'pvalues', 'metadata'
print('LR result means shape:', res['means'].shape)

In [ ]:
# Plot top significant interactions for selected cluster pairs
sq.pl.ligrec(
    res,
    source_groups=['Hippocampus'],
    alpha=0.001,
    means_range=(0.3, float('inf')),
    figsize=(8, 5),
)

---
## Summary

| Function | What it computes | Result location |
|----------|------------------|-----------------|
| `sq.gr.spatial_neighbors` | Spatial kNN graph | `.obsp['spatial_connectivities']` |
| `sq.gr.nhood_enrichment` | Co-localization z-scores | `.uns[f'{key}_nhood_enrichment']` |
| `sq.gr.co_occurrence` | Distance-resolved co-occurrence | `.uns[f'{key}_co_occurrence']` |
| `sq.gr.spatial_autocorr` | Moran's I per gene | `.uns['moranI']` / `.uns['gearyC']` |
| `sq.gr.ligrec` | LR permutation test | returned dict |
| `sq.pl.spatial_scatter` | Spots on tissue image | — |

**Next:** T03 — CellRank 2 for trajectory inference.